# LeetCode 2180 — Count Integers With Even Digit Sum bằng Python

**Link đề:** https://leetcode.com/problems/count-integers-with-even-digit-sum/
**Độ khó:** Easy · **Chủ đề:** Math, Simulation

```python
class Solution:
    def countEven(self, num: int) -> int:
        ...
```

## Mục tiêu học
1. Tính **tổng chữ số (digit sum)** bằng hai cách: chuỗi (`str`, `int`) và toán (`% 10`, `// 10`).
2. Brute force: duyệt `1..num`, đếm số có tổng chữ số chẵn — `O(num · log num)`.
3. Quan sát quy luật **chẵn/lẻ xen kẽ** giữa `2t` và `2t + 1` → công thức `O(log num)`:
   `answer = (num - digit_sum(num) % 2) // 2`.
4. Chứng minh công thức bằng **ghép cặp**, kiểm chứng trên toàn bộ miền input `1..1000`.
5. Làm quen ý tưởng **đếm theo khối 10** — bước đệm cho kỹ thuật digit DP.

## 1. Đề bài

Cho số nguyên dương `num`. Đếm số nguyên **dương** `x` với `x <= num` sao cho **tổng các chữ số** của `x` là số **chẵn**.

| `num` | Kết quả | Các số thỏa |
|:-:|:-:|---|
| `4` | `2` | 2, 4 |
| `30` | `14` | 2, 4, 6, 8, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28 |

**Constraints:** `1 <= num <= 1000`.

Chú ý: chỉ đếm số **dương** (bắt đầu từ 1). Số `0` có tổng chữ số `0` (chẵn) nhưng **không** được đếm.

In [1]:
num = 30
good = [x for x in range(1, num + 1) if sum(int(c) for c in str(x)) % 2 == 0]
print(good, "-> đếm được", len(good))

[2, 4, 6, 8, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28] -> đếm được 14


## 2. Nền tảng Python cần biết

### 2.1. Tổng chữ số bằng chuỗi

`str(x)` biến số thành chuỗi; duyệt từng ký tự, `int(c)` đổi về chữ số; `sum(...)` cộng lại.

In [2]:
x = 2938
print([int(c) for c in str(x)])
print("tổng chữ số:", sum(int(c) for c in str(x)))

[2, 9, 3, 8]
tổng chữ số: 22


### 2.2. Tổng chữ số bằng toán: `% 10` và `// 10`

- `x % 10` → chữ số cuối.
- `x // 10` → bỏ chữ số cuối.
- Lặp đến khi `x == 0`.

In [3]:
def digit_sum(x: int) -> int:
    s = 0
    while x > 0:
        s += x % 10
        x //= 10
    return s

for x in (7, 19, 2938, 1000):
    print(x, "->", digit_sum(x))

7 -> 7
19 -> 10
2938 -> 22
1000 -> 1


### 2.3. Kiểm tra chẵn: `% 2 == 0`

`s % 2 == 0` ⟺ `s` chẵn. Còn `s % 2` cho `0` (chẵn) hoặc `1` (lẻ) — ta sẽ dùng trực tiếp con số này trong công thức.

In [4]:
for s in (0, 3, 4, 11):
    print(s, "% 2 =", s % 2, "->", "chẵn" if s % 2 == 0 else "lẻ")

0 % 2 = 0 -> chẵn
3 % 2 = 1 -> lẻ
4 % 2 = 0 -> chẵn
11 % 2 = 1 -> lẻ


### 2.4. `divmod` và chia lấy phần nguyên `//`

`divmod(x, 10)` trả `(x // 10, x % 10)` cùng lúc. `//` luôn làm tròn **xuống**: `29 // 2 = 14`.

In [5]:
print(divmod(2938, 10), 29 // 2, 30 // 2)

(293, 8) 14 15


### 2.5. `map(hàm, dãy)` — áp dụng một hàm lên từng phần tử

`map(int, "2938")` tạo một iterator "lười" cho `int('2'), int('9'), ...`. Kết hợp `sum(map(int, str(x)))` là cách ngắn nhất để tính tổng chữ số. Muốn xem nội dung thì bọc `list(...)`.

In [6]:
m = map(int, "2938")
print(m)                 # chỉ là đối tượng map, chưa tính
print(list(m))           # [2, 9, 3, 8]
print(sum(map(int, str(2938))))

[2, 9, 3, 8]
22


### 2.6. Hiểu bài bằng tay — bảng chẵn/lẻ của tổng chữ số

Viết tổng chữ số theo hàng (mỗi hàng là một "khối 10" số):

In [7]:
for tens in range(0, 4):
    row = []
    for d in range(10):
        x = tens * 10 + d
        row.append("C" if digit_sum(x) % 2 == 0 else ".")
    print(f"{tens * 10:>2}..{tens * 10 + 9:<2}: {' '.join(row)}")
print("C = tổng chữ số chẵn, . = lẻ")

 0..9 : C . C . C . C . C .
10..19: . C . C . C . C . C
20..29: C . C . C . C . C .
30..39: . C . C . C . C . C
C = tổng chữ số chẵn, . = lẻ


Quan sát: trong **mỗi hàng** (khối 10 số), chẵn/lẻ **xen kẽ** nhau, nên mỗi khối có đúng **5** số tổng chẵn. Lý do: trong một khối, chỉ chữ số hàng đơn vị thay đổi, tăng 1 → tổng chữ số tăng 1 → đổi parity (tính chẵn lẻ). Ta sẽ khai thác điều này.

## 3. Cách 1 — Brute force bằng chuỗi

Đúng như hint của đề: duyệt `1..num`, tính tổng chữ số, đếm số chẵn.

In [8]:
def count_even_str(num: int) -> int:
    count = 0
    for x in range(1, num + 1):
        if sum(int(c) for c in str(x)) % 2 == 0:
            count += 1
    return count

print(count_even_str(4), count_even_str(30), count_even_str(1000))

2 14 499


## 4. Cách 2 — Brute force bằng toán (không dùng chuỗi)

In [9]:
def count_even_math(num: int) -> int:
    count = 0
    for x in range(1, num + 1):
        if digit_sum(x) % 2 == 0:
            count += 1
    return count

print(count_even_math(4), count_even_math(30), count_even_math(1000))

2 14 499


### Chi phí của brute force

Duyệt `num` số, mỗi số có khoảng `log10(num) + 1` chữ số → `O(num · log num)`. Với `num ≤ 1000` chỉ ~4000 phép tính — rất nhanh. **Brute force là lời giải chấp nhận được** cho bài này.

Nhưng nếu `num` lên tới `10^18`, ta không thể duyệt từng số. Câu hỏi: có **quy luật** nào không?

In [10]:
for num in range(1, 31):
    print(f"num={num:>2}: đáp án={count_even_math(num):>2}, num//2={num // 2:>2}, digit_sum(num) chẵn? {digit_sum(num) % 2 == 0}")

num= 1: đáp án= 0, num//2= 0, digit_sum(num) chẵn? False
num= 2: đáp án= 1, num//2= 1, digit_sum(num) chẵn? True
num= 3: đáp án= 1, num//2= 1, digit_sum(num) chẵn? False
num= 4: đáp án= 2, num//2= 2, digit_sum(num) chẵn? True
num= 5: đáp án= 2, num//2= 2, digit_sum(num) chẵn? False
num= 6: đáp án= 3, num//2= 3, digit_sum(num) chẵn? True
num= 7: đáp án= 3, num//2= 3, digit_sum(num) chẵn? False
num= 8: đáp án= 4, num//2= 4, digit_sum(num) chẵn? True
num= 9: đáp án= 4, num//2= 4, digit_sum(num) chẵn? False
num=10: đáp án= 4, num//2= 5, digit_sum(num) chẵn? False
num=11: đáp án= 5, num//2= 5, digit_sum(num) chẵn? True
num=12: đáp án= 5, num//2= 6, digit_sum(num) chẵn? False
num=13: đáp án= 6, num//2= 6, digit_sum(num) chẵn? True
num=14: đáp án= 6, num//2= 7, digit_sum(num) chẵn? False
num=15: đáp án= 7, num//2= 7, digit_sum(num) chẵn? True
num=16: đáp án= 7, num//2= 8, digit_sum(num) chẵn? False
num=17: đáp án= 8, num//2= 8, digit_sum(num) chẵn? True
num=18: đáp án= 8, num//2= 9, digit_sum

## 5. Insight: ghép cặp `(2t, 2t + 1)`

Từ bảng: đáp án luôn **xấp xỉ một nửa** `num` (bằng `num // 2` hoặc `num // 2 - 1`). Để biết chính xác khi nào là trường hợp nào, ta ghép cặp:

**Ghép cặp:** xét các cặp `(0,1), (2,3), (4,5), ..., (2t, 2t+1)`. Hai số trong một cặp chỉ khác nhau ở chữ số hàng đơn vị (`2t` tận cùng chẵn ≤ 8, cộng 1 không gây nhớ) → tổng chữ số khác nhau đúng 1 → **đúng một** số trong cặp có tổng chẵn.

Từ đó, với các số `0..num`:
- **`num` lẻ:** `0..num` chia đủ thành `(num + 1) / 2` cặp → có `(num + 1)/2` số tổng chẵn (kể cả số 0) → bỏ số 0: `(num - 1) / 2`.
- **`num` chẵn:** `0..num-1` thành `num / 2` cặp → `num / 2` số chẵn; cộng thêm `num` nếu `digit_sum(num)` chẵn; bỏ số 0 → `num/2 - (digit_sum(num) % 2)`.

Gộp hai trường hợp: **`answer = (num - digit_sum(num) % 2) // 2`**.

In [11]:
for num in (4, 30, 11, 10, 1000):
    p = digit_sum(num) % 2
    print(f"num={num}: digit_sum={digit_sum(num)}, p={p}, (num - p)//2 = {(num - p) // 2}, brute force = {count_even_math(num)}")

num=4: digit_sum=4, p=0, (num - p)//2 = 2, brute force = 2
num=30: digit_sum=3, p=1, (num - p)//2 = 14, brute force = 14
num=11: digit_sum=2, p=0, (num - p)//2 = 5, brute force = 5
num=10: digit_sum=1, p=1, (num - p)//2 = 4, brute force = 4
num=1000: digit_sum=1, p=1, (num - p)//2 = 499, brute force = 499


## 6. Cách 3 — Lời giải chính: công thức `O(log num)`

Từng đoạn:

**Đoạn 1:** `s = digit_sum(num)` — `O(số chữ số)`.

**Đoạn 2:** `p = s % 2` (0 nếu chẵn, 1 nếu lẻ).

**Đoạn 3:** trả `(num - p) // 2`.

In [12]:
def count_even_formula(num: int) -> int:
    s = 0
    x = num
    while x > 0:                 # Đoạn 1
        s += x % 10
        x //= 10
    p = s % 2                    # Đoạn 2
    return (num - p) // 2        # Đoạn 3

print(count_even_formula(4), count_even_formula(30), count_even_formula(1000))

2 14 499


## 7. TRACE từng bước

**Trace công thức với `num = 30`:**

| Bước | `x` | `x % 10` | `s` sau |
|:-:|:-:|:-:|:-:|
| 1 | 30 | 0 | 0 |
| 2 | 3 | 3 | 3 |
| dừng | 0 | | **3** |

`p = 3 % 2 = 1` → `(30 - 1) // 2 = 29 // 2 = 14`. ✓

**Trace ghép cặp** để thấy vì sao: in các cặp `(2t, 2t+1)` cùng parity tổng chữ số.

In [13]:
def trace_formula(num):
    s, x, step = 0, num, 1
    while x > 0:
        s += x % 10
        print(f"Bước {step}: x={x:>4}, chữ số cuối {x % 10}, s={s}")
        x //= 10
        step += 1
    p = s % 2
    print(f"p = {s} % 2 = {p} -> ({num} - {p}) // 2 = {(num - p) // 2}")
    return (num - p) // 2

trace_formula(30)

Bước 1: x=  30, chữ số cuối 0, s=0
Bước 2: x=   3, chữ số cuối 3, s=3
p = 3 % 2 = 1 -> (30 - 1) // 2 = 14


14

In [14]:
def trace_pairs(num):
    evens = []
    for t in range(0, num + 1, 2):
        a, b_ = t, t + 1
        pa = digit_sum(a) % 2
        pb = digit_sum(b_) % 2 if b_ <= num else None
        chosen = [v for v, pv in ((a, pa), (b_, pb)) if pv == 0 and v <= num]
        evens += chosen
        print(f"cặp ({a:>2}, {b_:>2}): parity {pa}, {pb} -> số chẵn trong cặp: {chosen}")
    positive = [v for v in evens if v > 0]
    print("bỏ số 0 ->", len(positive), "số:", positive)

trace_pairs(12)

cặp ( 0,  1): parity 0, 1 -> số chẵn trong cặp: [0]
cặp ( 2,  3): parity 0, 1 -> số chẵn trong cặp: [2]
cặp ( 4,  5): parity 0, 1 -> số chẵn trong cặp: [4]
cặp ( 6,  7): parity 0, 1 -> số chẵn trong cặp: [6]
cặp ( 8,  9): parity 0, 1 -> số chẵn trong cặp: [8]
cặp (10, 11): parity 1, 0 -> số chẵn trong cặp: [11]
cặp (12, 13): parity 1, None -> số chẵn trong cặp: []
bỏ số 0 -> 5 số: [2, 4, 6, 8, 11]


Cặp `(12, 13)` có `13 > num = 12` nên chỉ còn `12` (tổng chữ số 3, lẻ) → không đóng góp. Đáp án `num=12`: `5` số `{2, 4, 6, 8, 11}`; công thức: `digit_sum(12)=3`, `p=1`, `(12-1)//2 = 5`. ✓

## 8. Code nộp LeetCode

**Bản brute force** (đủ nhanh với `num ≤ 1000`):

In [15]:
class Solution:
    def countEven(self, num: int) -> int:
        return sum(1 for x in range(1, num + 1) if sum(map(int, str(x))) % 2 == 0)

print(Solution().countEven(4), Solution().countEven(30))

2 14


`map(int, str(x))` áp dụng `int` lên từng ký tự — tương đương `(int(c) for c in str(x))`.

**Bản công thức** (tối ưu):

In [16]:
class Solution:
    def countEven(self, num: int) -> int:
        s = sum(map(int, str(num)))
        return (num - s % 2) // 2

sol = Solution()
print(sol.countEven(4), sol.countEven(30), sol.countEven(1), sol.countEven(1000))

2 14 0 499


## 9. Cách 4 — Đếm theo khối 10 (nền móng cho digit DP)

Một cách nghĩ khác, không cần mẹo ghép cặp, tổng quát hơn:
- Gọi `q = num // 10`, `r = num % 10`. Các số `0..10q - 1` chia thành `q` khối 10 đầy đủ, mỗi khối có **5** số tổng chẵn → `5q`.
- Khối cuối `10q, 10q + 1, ..., 10q + r`: tổng chữ số là `digit_sum(q) + d` với `d = 0..r`. Đếm `d` sao cho `digit_sum(q) + d` chẵn.
- Trừ 1 vì số `0` bị tính nhưng không dương.

In [17]:
def count_even_blocks(num: int) -> int:
    q, r = divmod(num, 10)
    full = 5 * q
    base = digit_sum(q) % 2
    partial = sum(1 for d in range(r + 1) if (base + d) % 2 == 0)
    return full + partial - 1          # bỏ số 0

print(count_even_blocks(4), count_even_blocks(30), count_even_blocks(1000))

2 14 499


In [18]:
def trace_blocks(num):
    q, r = divmod(num, 10)
    base = digit_sum(q) % 2
    part = [10 * q + d for d in range(r + 1) if (base + d) % 2 == 0]
    print(f"num={num}: q={q} khối đầy đủ -> 5*{q} = {5 * q}")
    print(f"  khối cuối {10 * q}..{num}: digit_sum({q}) parity {base}, các số chẵn {part} -> {len(part)}")
    print(f"  tổng = {5 * q} + {len(part)} - 1 (bỏ số 0) = {5 * q + len(part) - 1}")

trace_blocks(30)
trace_blocks(47)

num=30: q=3 khối đầy đủ -> 5*3 = 15
  khối cuối 30..30: digit_sum(3) parity 1, các số chẵn [] -> 0
  tổng = 15 + 0 - 1 (bỏ số 0) = 14
num=47: q=4 khối đầy đủ -> 5*4 = 20
  khối cuối 40..47: digit_sum(4) parity 0, các số chẵn [40, 42, 44, 46] -> 4
  tổng = 20 + 4 - 1 (bỏ số 0) = 23


Ý tưởng "xử lý phần đầy đủ bằng công thức, phần cuối dở dang riêng" chính là tinh thần của **digit DP** — kỹ thuật đếm số `≤ N` thỏa điều kiện về chữ số, dùng cho các bài khó hơn.

## 10. Vì sao đúng? (chứng minh)

**Bổ đề ghép cặp:** với mọi `t ≥ 0`, `digit_sum(2t + 1) = digit_sum(2t) + 1`.
Vì `2t` tận cùng bằng chữ số chẵn `0, 2, 4, 6, 8`, cộng 1 chỉ tăng chữ số cuối (không nhớ sang hàng chục). ⇒ Đúng một trong hai số `2t`, `2t+1` có tổng chữ số chẵn.

**Đếm trên `0..num`:**
- `num` lẻ: có đúng `(num+1)/2` cặp phủ kín → `(num+1)/2` số chẵn (gồm 0) → đáp án `(num-1)/2 = (num - p)//2` với mọi `p ∈ {0,1}` (vì `num` lẻ).
- `num` chẵn: `num/2` cặp phủ `0..num-1`, cộng số `num` nếu `p = 0` → `num/2 + 1 - p` (gồm 0) → đáp án `num/2 - p = (num - p)//2` (với `num` chẵn: `p=0` → `num/2`; `p=1` → `(num-1)//2 = num/2 - 1`).

**Cách 4:** khối `[10a, 10a+9]` có tổng chữ số `digit_sum(a) + d`, `d = 0..9` → 5 chẵn, 5 lẻ. Khối cuối đếm trực tiếp. Trừ số 0.

Phần test sẽ kiểm tra **toàn bộ** `1..1000` (và xa hơn) để chắc chắn.

## 11. Độ phức tạp

Gọi `D = số chữ số của num ≈ log10(num) + 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force chuỗi | `O(num · D)` | `O(D)` (chuỗi tạm) | dễ nhất |
| 2. Brute force toán | `O(num · D)` | `O(1)` | không tạo chuỗi |
| 3. Công thức | `O(D)` | `O(1)` hoặc `O(D)` nếu dùng `str` | lời giải chính |
| 4. Khối 10 | `O(D)` | `O(1)` | tổng quát hóa được |

Với `num = 10^18`: brute force cần `10^18` bước (không thể), công thức chỉ ~19 bước.

### So sánh tốc độ thực tế (minh họa)

Đo thời gian với `num = 10^6` (ngoài constraint, chỉ để thấy khác biệt độ phức tạp; con số phụ thuộc máy):

In [19]:
import time
N = 10**6
t0 = time.perf_counter(); r1 = count_even_math(N); t1 = time.perf_counter()
r2 = count_even_formula(N); t2 = time.perf_counter()
print(f"brute force: {r1} trong {t1 - t0:.3f}s | công thức: {r2} trong {(t2 - t1) * 1e6:.1f} micro giây")
assert r1 == r2

brute force: 499999 trong 0.432s | công thức: 499999 trong 51.0 micro giây


## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Hai ví dụ của đề.
2. `num = 1` (số 1 có tổng lẻ → 0), `num = 2`.
3. `num` chẵn và tổng chữ số chẵn / lẻ; `num` lẻ.
4. Mốc chuyển hàng: `9, 10, 11, 99, 100, 1000`.
5. **Toàn bộ** `1..1000` so với brute force; thêm vùng lớn hơn cho công thức.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

impls = [count_even_str, count_even_math, count_even_formula, count_even_blocks, Solution().countEven]
cases = [
    ("đề ví dụ 1", 4, 2),
    ("đề ví dụ 2", 30, 14),
    ("num = 1", 1, 0),
    ("num = 2", 2, 1),
    ("num = 9", 9, 4),
    ("num = 10", 10, 4),
    ("num = 11", 11, 5),
    ("num = 99", 99, 49),
    ("num = 100", 100, 49),
    ("num = 1000", 1000, 499),
]
for name, num, exp in cases:
    for f in impls:
        assert f(num) == exp, (name, f)
    check(name, count_even_formula(num), exp)

đề ví dụ 1             PASS (2)
đề ví dụ 2             PASS (14)
num = 1                PASS (0)
num = 2                PASS (1)
num = 9                PASS (4)
num = 10               PASS (4)
num = 11               PASS (5)
num = 99               PASS (49)
num = 100              PASS (49)
num = 1000             PASS (499)


In [21]:
running = 0
for num in range(1, 5001):
    if digit_sum(num) % 2 == 0:
        running += 1                       # đếm dồn = brute force nhanh
    assert count_even_formula(num) == running == count_even_blocks(num), num
    if num <= 1000:
        assert count_even_str(num) == running
print("Đã kiểm tra toàn bộ 1..5000 (brute force đầy đủ trên 1..1000): PASS")

Đã kiểm tra toàn bộ 1..5000 (brute force đầy đủ trên 1..1000): PASS


Mẹo kiểm thử: thay vì gọi brute force `O(num)` cho từng `num` (tổng `O(num²)`), ta **đếm dồn** (`running`) — mỗi bước chỉ xét thêm một số.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Đếm cả số 0

In [22]:
def wrong_include_zero(num):
    return sum(1 for x in range(0, num + 1) if digit_sum(x) % 2 == 0)

print("sai :", wrong_include_zero(4), "| đúng:", count_even_math(4))

sai : 3 | đúng: 2


**Vì sao sai:** đề yêu cầu số **dương**. **Sửa:** `range(1, num + 1)`.

### Lỗi 2: `range(1, num)` bỏ sót chính `num`

In [23]:
def wrong_exclusive(num):
    return sum(1 for x in range(1, num) if digit_sum(x) % 2 == 0)

print("sai :", wrong_exclusive(4), "| đúng:", count_even_math(4))

sai : 1 | đúng: 2


**Vì sao sai:** `range(a, b)` không gồm `b`; đề nói `x <= num`. **Sửa:** `range(1, num + 1)`.

### Lỗi 3: Nhầm "tổng chữ số chẵn" với "số chẵn"

In [24]:
def wrong_even_number(num):
    return sum(1 for x in range(1, num + 1) if x % 2 == 0)

print("sai :", wrong_even_number(30), "| đúng:", count_even_math(30))

sai : 15 | đúng: 14


**Vì sao sai:** `11` là số lẻ nhưng tổng chữ số `2` chẵn; `10` là số chẵn nhưng tổng chữ số `1` lẻ. Với `num = 30` hai đáp án tình cờ gần nhau (15 vs 14) nên rất dễ không phát hiện! **Sửa:** kiểm tra `digit_sum(x) % 2`.

### Lỗi 4: Công thức "luôn là `num // 2`"

In [25]:
for num in (4, 30, 12, 10):
    print(num, "num//2 =", num // 2, "| đúng:", count_even_formula(num))

4 num//2 = 2 | đúng: 2
30 num//2 = 15 | đúng: 14
12 num//2 = 6 | đúng: 5
10 num//2 = 5 | đúng: 4


**Vì sao sai:** khi `num` chẵn và tổng chữ số của `num` **lẻ** (ví dụ `10`, `12`, `30` — đều là số chẵn nhưng tổng chữ số lẻ), đáp án là `num//2 - 1`. **Sửa:** dùng `(num - digit_sum(num) % 2) // 2`.

### Lỗi 5: Dùng `/` thay vì `//`

In [26]:
num = 30
print((num - 1) / 2, "<- float", (num - 1) // 2, "<- int đúng")

14.5 <- float 14 <- int đúng


## 14. Mẹo nhận dạng pattern

- "Tổng chữ số", "chữ số của x" → `% 10` / `// 10` hoặc `str` + `int`.
- "Đếm số `≤ N` thỏa điều kiện chẵn/lẻ" → thử in bảng nhỏ, tìm **quy luật xen kẽ** → ghép cặp/khối.
- Miền nhỏ (`≤ 1000`) → brute force đủ; luôn giữ brute force làm oracle khi tìm công thức.
- Bài "đếm số `≤ N` theo chữ số" tổng quát → **digit DP** (khối đầy đủ + phần dở dang).

Bài liên quan: #1295 Find Numbers with Even Number of Digits, #258 Add Digits, #1281 Subtract the Product and Sum of Digits, #2553 Separate the Digits in an Array, #233 Number of Digit One (digit DP, Hard).

## 15. Biến thể

### 15.1. Đếm số có tổng chữ số **chia hết cho `m`** trong `1..num` (brute force + đếm dồn)

In [27]:
def count_digit_sum_divisible(num, m):
    return sum(1 for x in range(1, num + 1) if digit_sum(x) % m == 0)

print(count_digit_sum_divisible(30, 2), count_digit_sum_divisible(100, 3), count_digit_sum_divisible(100, 9))

14 33 11


### 15.2. Digit DP nhỏ: đếm số trong `0..N` có tổng chữ số chẵn (cho `N` khổng lồ)
Duyệt chữ số từ trái sang phải với trạng thái `(vị trí, parity, đang bám sát N?)`. Dùng `functools.lru_cache` để nhớ kết quả (memoization).

In [28]:
from functools import lru_cache

def count_even_digit_dp(N: int) -> int:
    digits = list(map(int, str(N)))

    @lru_cache(maxsize=None)
    def go(pos, parity, tight):
        if pos == len(digits):
            return 1 if parity == 0 else 0
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            total += go(pos + 1, (parity + d) % 2, tight and d == limit)
        return total

    return go(0, 0, True) - 1          # bỏ số 0

print(count_even_digit_dp(30), count_even_digit_dp(1000))
big = 10**18 + 7
print(count_even_digit_dp(big), count_even_formula(big), count_even_digit_dp(big) == count_even_formula(big))

14 499
500000000000000003 500000000000000003 True


`@lru_cache` là một **decorator**: bọc hàm `go` để lưu kết quả theo tham số, gọi lại cùng tham số thì trả ngay. Digit DP chạy `O(D · 2 · 2 · 10)` — tổng quát cho mọi điều kiện kiểu "tổng chữ số mod m".

## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Tính tay
`countEven(25)` = ? Dùng công thức và kiểm tra bằng cách liệt kê.

### Bài 2 — Đoán output
1. `digit_sum(1000)` = ?
2. `(1000 - digit_sum(1000) % 2) // 2` = ?
3. `sum(map(int, str(909)))` = ?

### Bài 3 — Điền chỗ trống (hàm tổng chữ số)

In [29]:
def exercise_digit_sum(x: int) -> int:
    s = 0
    # TODO 1: while x > 0
    # TODO 2: cộng chữ số cuối vào s
    # TODO 3: bỏ chữ số cuối
    return s

# assert exercise_digit_sum(2938) == 22
# assert exercise_digit_sum(1000) == 1

### Bài 4 — Tự viết `countEven` bằng công thức

In [30]:
def exercise_count_even(num: int) -> int:
    # TODO
    pass

# assert exercise_count_even(4) == 2
# assert exercise_count_even(30) == 14

### Bài 5 — Biến thể: đếm số có tổng chữ số **lẻ** trong `1..num`
Gợi ý: không cần viết lại từ đầu.

In [31]:
def exercise_count_odd(num: int) -> int:
    # TODO
    pass

# assert exercise_count_odd(4) == 2      # 1, 3
# assert exercise_count_odd(30) == 16

### Bài 6 — Mini challenge: đếm trong **đoạn** `[lo, hi]`
Đếm số `x` với `lo <= x <= hi` có tổng chữ số chẵn, `O(log hi)`.

In [32]:
def exercise_count_even_range(lo: int, hi: int) -> int:
    # TODO: dùng hiệu hai tiền tố
    pass

# assert exercise_count_even_range(1, 30) == 14
# assert exercise_count_even_range(11, 20) == 6     # 11, 13, 15, 17, 19, 20

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tổng chữ số của 25 | `2 + 5 = 7` lẻ | `(25 - 1) // 2` |
| 2 | Chỉ một chữ số 1 | `p = 1` | `909` → 9+0+9 |
| 3 | `x % 10` | `s += x % 10` | `x //= 10` |
| 4 | Tính `p = digit_sum(num) % 2` | `(num - p) // 2` | dùng `sum(map(int, str(num)))` |
| 5 | Mọi số dương ≤ num có tổng hoặc chẵn hoặc lẻ | `num - countEven(num)` | tái sử dụng hàm của Bài 4 |
| 6 | `f(hi) - f(lo - 1)` | `f(0) = 0` | `f` là hàm `countEven` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `digit_sum(25) = 7`, `p = 1` → `(25 - 1) // 2 = 12`. Liệt kê: 2, 4, 6, 8, 11, 13, 15, 17, 19, 20, 22, 24 → 12 số. ✓

**Bài 2:** `1`; `(1000 - 1) // 2 = 499`; `18`.

In [33]:
print(count_even_formula(25), [x for x in range(1, 26) if digit_sum(x) % 2 == 0])
print(digit_sum(1000), (1000 - digit_sum(1000) % 2) // 2, sum(map(int, str(909))))

12 [2, 4, 6, 8, 11, 13, 15, 17, 19, 20, 22, 24]
1 499 18


In [34]:
def answer_digit_sum(x):
    s = 0
    while x > 0:
        s += x % 10
        x //= 10
    return s


def answer_count_even(num):
    return (num - answer_digit_sum(num) % 2) // 2


def answer_count_odd(num):
    return num - answer_count_even(num)


def answer_count_even_range(lo, hi):
    f = lambda n: answer_count_even(n) if n > 0 else 0
    return f(hi) - f(lo - 1)


assert answer_digit_sum(2938) == 22
assert answer_digit_sum(1000) == 1
assert answer_count_even(4) == 2
assert answer_count_even(30) == 14
assert answer_count_odd(4) == 2
assert answer_count_odd(30) == 16
assert answer_count_even_range(1, 30) == 14
assert answer_count_even_range(11, 20) == 6
for lo in range(1, 60):
    for hi in range(lo, 120):
        assert answer_count_even_range(lo, hi) == sum(1 for x in range(lo, hi + 1) if digit_sum(x) % 2 == 0)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countEven(self, num: int) -> int:
        s = sum(map(int, str(num)))      # tổng chữ số của num
        return (num - s % 2) // 2
```

Brute force an toàn (đủ nhanh với `num ≤ 1000`):
```python
return sum(1 for x in range(1, num + 1) if sum(map(int, str(x))) % 2 == 0)
```

**Khung tư duy:** brute force + in bảng nhỏ → phát hiện quy luật (xen kẽ, khối 10) → chứng minh bằng ghép cặp → công thức → kiểm chứng toàn miền.

**Checklist trước khi nộp:**
- [ ] Chỉ đếm số **dương** (bỏ 0)?
- [ ] Gồm cả `num` (`range(1, num + 1)`)?
- [ ] Kiểm tra **tổng chữ số** chẵn, không phải số chẵn?
- [ ] Dùng `//` (số nguyên)?
- [ ] Test `1, 2, 10, 11, 1000`.